# Gaia Star Saturation – Version locale via API TAP (pyvo)

- **Auteur** : Sylvie Dagoret-Campagne  
- **Creation Date** : 2026-02-28
- **Last Update** : 2026-03-03
- **Objectif** : Adaptation locale du notebook RSP `gaiaStarSaturation_workaround.ipynb`

## Différences vs version RSP

| RSP | Local |
|-----|-------|
| `lsst.rsp.get_tap_service` | `pyvo` + token `RSP_TOKEN` |
| `lsst.daf.butler` | Non disponible – partie Butler supprimée |
| `lsst.utils.plotting` | matplotlib standard |
| 3 requêtes séparées + `pd.merge` | **JOIN ADQL unique** Source ⋈ Visit ⋈ CcdVisit |
| `afwDisplay` (Firefly) | `matplotlib` + `astropy.visualization` |

## Pré-requis
```bash
pip install pyvo astropy matplotlib numpy pandas scipy scikit-learn seaborn
export RSP_TOKEN="votre_token_ici"  # dans ~/.zshrc ou ~/.bashrc
```

## 1. Imports

In [ ]:
import os
import importlib
import pprint

import matplotlib.pyplot as plt
%matplotlib inline
import matplotlib.gridspec as gridspec
import matplotlib.patheffects as pathEffects
from matplotlib import colors
from matplotlib.ticker import PercentFormatter

import math
import numpy as np
import pandas as pd

from scipy.stats import binned_statistic, binned_statistic_2d, median_abs_deviation, gaussian_kde
from scipy.optimize import curve_fit
from scipy import stats, linalg
from scipy.stats import linregress

import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.table import Table, vstack, hstack
from astropy.stats import sigma_clipped_stats
from astropy.visualization import (
    LinearStretch, ZScaleInterval, imshow_norm
)

from sklearn.linear_model import LinearRegression, RANSACRegressor

import pyvo
import requests

from pathlib import Path

## 2. Connexion au service TAP de Rubin DP1

In [ ]:
# Token stocké dans la variable d'environnement RSP_TOKEN
# export RSP_TOKEN="votre_token" dans ton .zshrc ou .bashrc
token = os.environ["RSP_TOKEN"]

# Session HTTP authentifiée
session = requests.Session()
session.headers["Authorization"] = f"Bearer {token}"

RSP_TAP_URL = "https://data.lsst.cloud/api/tap"
service = pyvo.dal.TAPService(RSP_TAP_URL, session=session)
print("Service TAP connecté :", RSP_TAP_URL)

In [ ]:
# Helper : soumettre et attendre une requête asynchrone
def run_tap_query(service, query, verbose=True):
    """Soumet un job TAP asynchrone, attend le résultat et retourne un DataFrame pandas."""
    job = service.submit_job(query)
    job.run()
    job.wait(phases=["COMPLETED", "ERROR"])
    if verbose:
        print("Job phase:", job.phase)
    if job.phase == "ERROR":
        job.raise_if_error()
    table = job.fetch_result().to_table()
    if verbose:
        print(f"  → {len(table)} lignes retournées")
    return table.to_pandas()

## 3. Paramètres de la région d'intérêt (ECDFS)

In [ ]:
# Centre ECDFS
TAG_FIELD = "ECDFS"
RA_CENTER   = 53.0    # degrés
DEC_CENTER  = -28.0   # degrés
#RADIUS_DEG  = 1.75    # rayon en degrés
RADIUS_DEG  = 0.75 

# Limite de magnitude LSST
LSST_MAG_LIMIT = 20.5
FLUX_LIMIT = 10**(-0.4 * (LSST_MAG_LIMIT - 31.4))  # en nJy

BANDS = "'g', 'r'"

print(f"Région : RA={RA_CENTER}, Dec={DEC_CENTER}, R={RADIUS_DEG}°")
print(f"Flux limite : {FLUX_LIMIT:.1f} nJy (mag < {LSST_MAG_LIMIT})")

## 4. Requête principale : Source JOIN Visit JOIN CcdVisit (ADQL)

Le JOIN est fait **côté serveur** directement en ADQL, ce qui évite les 3 requêtes séparées + merge Python du notebook RSP.

- `dp1.Source` → mesures individuelles (psfFlux, visit, detector, ra, dec, band)
- `dp1.Visit` → métadonnées de visite (expMidptMJD, airmass, band, ra/dec pointing…)
- `dp1.CcdVisit` → métadonnées par CCD (seeing, skyBg, zeroPoint, magLim, skyNoise…)

In [ ]:
#try:
#    res2 = service.search("""
#        SELECT TOP 5
#            src.sourceId, src.Visit, src.band,
#            src.psfFlux, src.psfFluxErr,
#            v.expMidptMJD, v.airmass,
#            cv.VisitId, cv.seeing, cv.zeroPoint, cv.magLim, cv.skyBg, cv.skyNoise
#        FROM dp1.Source AS src
#        JOIN dp1.Visit    AS v  ON src.Visit = v.visit
#        JOIN dp1.CcdVisit AS cv ON src.Visit = cv.VisitId
#        WHERE CONTAINS(
#            POINT('ICRS', src.coord_ra, src.coord_dec),
#            CIRCLE('ICRS', 53.13, -28.10, 0.05)
#        ) = 1
#        AND src.band = 'g'
#    """).to_table().to_pandas()
#    print(f"  OK - {len(res2)} lignes")
#    display(res2)
#except Exception as e:
#    print(f"  ERREUR : {e}")

In [ ]:
query_sources_joined = f"""
SELECT 
    src.sourceId,
    src.Visit,
    src.detector,
    src.ra,
    src.dec,
    src.psfFlux,
    src.psfFluxErr,
    src.band,
    -- Visit metadata
    vis.expMidptMJD,
    vis.airmass,
    vis.ra        AS visit_ra,
    vis.dec       AS visit_dec,
    vis.obsStart,
    -- CcdVisit metadata (qualité par CCD)
    ccd.seeing,
    ccd.skyBg,
    ccd.skyNoise,
    ccd.zeroPoint,
    ccd.magLim,
    ccd.obsStartMJD AS obsStartMJD_det
FROM
    dp1.Source AS src
    JOIN dp1.Visit    AS vis ON src.Visit    = vis.visit
    JOIN dp1.CcdVisit AS ccd ON src.Visit    = ccd.VisitId
WHERE
    CONTAINS(
        POINT('ICRS', src.coord_ra, src.coord_dec),
        CIRCLE('ICRS', {RA_CENTER}, {DEC_CENTER}, {RADIUS_DEG})
    ) = 1
    AND src.psfFlux > {FLUX_LIMIT}
    AND src.band IN ({BANDS})
ORDER BY src.psfFlux ASC
"""

print("Lancement de la requête JOIN Source × Visit × CcdVisit...")
print("(peut prendre quelques minutes)")

In [ ]:
#df_sources = run_tap_query(service, query_sources_joined)
#print(df_sources.shape)
#df_sources.head(3)

In [ ]:
SOURCES_FILE = f"sourcesCombined_{TAG_FIELD}.csv"
CSV_PATH = Path(SOURCES_FILE)
if CSV_PATH.exists():
    df_sources  = pd.read_csv(CSV_PATH)
    print(f"CSV chargé : {len( df_sources)} lignes")
    print(df_sources.columns.tolist())
else:
    df_sources = run_tap_query(service, query_sources_joined)
    print(df_sources.shape)
    df_sources.head(3)   
    df_sources.to_csv(SOURCES_FILE) 

### 4b. Optionnel : charger `table_for_visu_inspect.csv` si disponible localement

Si vous avez déjà exporté les données depuis la RSP (contenant les colonnes Gaia), vous pouvez les charger directement pour éviter une nouvelle requête TAP.

In [ ]:
CSV_PATH = Path("table_for_visu_inspect.csv")

if CSV_PATH.exists():
    stardata = pd.read_csv(CSV_PATH)
    print(f"CSV chargé : {len(stardata)} lignes")
    print(stardata.columns.tolist())
    USE_CSV = True
else:
    print("CSV non trouvé – on utilise df_sources (sans colonnes Gaia)")
    stardata = df_sources.copy()
    USE_CSV = False

## 5. Préparation des données

In [ ]:
# SNR et filtres de qualité
stardata['psfFluxSNR'] = stardata['psfFlux'] / stardata['psfFluxErr']
stardata = stardata[stardata['psfFluxSNR'] > 3.].copy()
stardata = stardata[stardata['band'] == 'g'].copy()

print(f"Après coupure SNR>3 et band=g : {len(stardata)} lignes")

In [ ]:
# Conversion flux → magnitude AB
stardata['psfFlux_mag'] = (stardata['psfFlux'].values * u.nanojansky).to(u.ABmag).value

mag_error_plus  = ((stardata['psfFlux'] + stardata['psfFluxErr']).values * u.nanojansky).to(u.ABmag).value
mag_error_minus = ((stardata['psfFlux'] - stardata['psfFluxErr']).values * u.nanojansky).to(u.ABmag).value
stardata['mag_error'] = 0.5 * (mag_error_minus - mag_error_plus)

if USE_CSV and 'gaia_bp_mag' in stardata.columns:
    stardata['gaia_color'] = stardata['gaia_bp_mag'] - stardata['gaia_rp_mag']
    stardata['delta_mag']  = stardata['mag'] - stardata['gaia_g_mag']
    HAS_GAIA = True
    print("Colonnes Gaia disponibles.")
else:
    HAS_GAIA = False
    print("Pas de colonnes Gaia – certaines sections seront skippées.")

## 6. Histogrammes exploratoires

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(stardata['psfFlux'], bins=80, log=True, color='steelblue')
axes[0].set_xlabel('psfFlux (nJy)')
axes[0].set_ylabel('N')
axes[0].set_title('Distribution psfFlux')

axes[1].hist(stardata['psfFluxSNR'], bins=80, log=True, color='darkorange')
axes[1].set_xlabel('SNR PSF Flux')
axes[1].set_title('Distribution SNR')

axes[2].hist(stardata['psfFlux_mag'], bins=80, color='seagreen')
axes[2].set_xlabel('PSF Mag AB')
axes[2].set_title('Distribution magnitudes')

plt.tight_layout()
plt.show()
print(f"Nombre de sources : {len(stardata)}")
print(f"Nombre de visites uniques : {stardata['visit'].nunique()}")

## 7. Exploitation des métadonnées Visit/CcdVisit (issues du JOIN)

Ces colonnes sont directement disponibles grâce au JOIN ADQL – pas besoin de merge Python.

In [ ]:
# Vérifier les colonnes issues du JOIN
visit_cols = ['expMidptMJD', 'airmass', 'visit_ra', 'visit_dec', 'obsStart']
ccd_cols   = ['seeing', 'skyBg', 'skyNoise', 'zeroPoint', 'magLim', 'obsStartMJD_det', 'azimuth']

print("Colonnes Visit disponibles :", [c for c in visit_cols if c in stardata.columns])
print("Colonnes CcdVisit disponibles :", [c for c in ccd_cols if c in stardata.columns])
print()
# Statistiques rapides
available_ccd = [c for c in ccd_cols if c in stardata.columns]
if available_ccd:
    stardata[available_ccd].describe()

In [ ]:
if 'airmass' in stardata.columns and 'seeing' in stardata.columns:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    axes[0].hist(stardata['airmass'].dropna(), bins=50, color='royalblue')
    axes[0].set_xlabel('Airmass')
    axes[0].set_title('Distribution airmass')
    
    axes[1].hist(stardata['seeing'].dropna(), bins=50, color='tomato')
    axes[1].set_xlabel('Seeing (arcsec)')
    axes[1].set_title('Distribution seeing')
    
    axes[2].hist(stardata['zeroPoint'].dropna(), bins=50, color='mediumseagreen')
    axes[2].set_xlabel('ZeroPoint (mag)')
    axes[2].set_title('Distribution zeroPoint')
    
    plt.tight_layout()
    plt.show()

## 8. Analyse Gaia (si données Gaia disponibles)

In [ ]:
if HAS_GAIA:
    plt.figure(figsize=(8, 5))
    sc = plt.scatter(stardata['gaia_g_mag'], stardata['delta_mag'],
                     c=stardata['gaia_color'], alpha=0.1, s=3, cmap='RdYlBu_r')
    plt.colorbar(sc, label='Gaia BP-RP color')
    plt.xlabel('Gaia g mag')
    plt.ylabel('LSST g mag – Gaia g mag')
    plt.title('Offset photométrique vs Gaia')
    plt.axhline(0, color='k', lw=0.8, ls='--')
    plt.grid(alpha=0.3)
    plt.show()
else:
    print("Section Gaia ignorée (pas de données Gaia dans le CSV).")

## 9. Correction RANSAC par visite (si Gaia disponible)

In [ ]:
if HAS_GAIA:
    stardata['g_ransac'] = np.nan
    
    for vn, agroup in stardata.groupby('visit'):
        df = agroup[agroup['mag'] > 17]
        df = df[np.abs(df['gaia_color']) < 1.5]
        df = df[(df['psfFlux'] / df['psfFluxErr']) > 10]
        if len(df) < 50:
            continue
        
        x = df['gaia_color'].values
        y = df['delta_mag'].values
        
        reg = RANSACRegressor(estimator=LinearRegression())
        reg.fit(x.reshape(-1, 1), y.reshape(-1, 1))
        slp = reg.estimator_.coef_[0]
        zp  = reg.estimator_.intercept_
        
        lsst_g = agroup['mag'].values - zp - slp * agroup['gaia_color'].values
        stardata.loc[agroup.index, 'g_ransac'] = lsst_g
    
    print(f"Correction RANSAC appliquée pour {stardata['g_ransac'].notna().sum()} étoiles")
else:
    print("Section RANSAC ignorée.")

## 10. Analyse de corrélation : delta_mag vs métadonnées observationnelles

Grâce au JOIN, toutes les colonnes sont disponibles directement dans `stardata`.

In [ ]:
# Colonnes pour la matrice de corrélation
# Adapter selon les colonnes disponibles
candidate_corr_cols = [
    'psfFluxSNR', 'airmass', 'seeing', 'skyNoise',
    'zeroPoint', 'skyBg', 'magLim', 'obsStartMJD_det', 'azimuth'
]
if HAS_GAIA:
    candidate_corr_cols = ['delta_mag'] + candidate_corr_cols

cols_to_corr = [c for c in candidate_corr_cols if c in stardata.columns]
print("Colonnes retenues pour corrélation :", cols_to_corr)

In [ ]:
if len(cols_to_corr) >= 2:
    try:
        import seaborn as sns
        
        subset = stardata[cols_to_corr].dropna()
        if HAS_GAIA and 'delta_mag' in subset.columns:
            subset = subset[subset['delta_mag'] > 0.2]
        
        corr = subset.corr()
        
        fig, ax = plt.subplots(figsize=(10, 8))
        sns.heatmap(
            corr,
            cmap=sns.diverging_palette(220, 10, as_cmap=True),
            vmin=-1.0, vmax=1.0,
            annot=True, fmt='.2f', annot_kws={'size': 8},
            square=True, ax=ax
        )
        ax.set_title('Matrice de corrélation (sources avec delta_mag > 0.2)')
        plt.tight_layout()
        plt.show()
    except ImportError:
        print("seaborn non installé – pip install seaborn")
        corr = stardata[cols_to_corr].dropna().corr()
        print(corr.round(3))

In [ ]:
# Scatter seeing vs delta_mag (coloré par airmass)
if HAS_GAIA and 'seeing' in stardata.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    sc = axes[0].scatter(
        stardata['seeing'], stardata['delta_mag'],
        c=stardata['airmass'], s=8, alpha=0.3, cmap='plasma'
    )
    plt.colorbar(sc, ax=axes[0], label='airmass')
    axes[0].set_xlabel('Seeing (arcsec)')
    axes[0].set_ylabel('delta_mag (LSST - Gaia g)')
    axes[0].set_title('Seeing vs delta_mag')
    axes[0].axhline(0, color='k', lw=0.8, ls='--')

    sc2 = axes[1].scatter(
        stardata['airmass'], stardata['delta_mag'],
        c=stardata['zeroPoint'], s=8, alpha=0.3, cmap='viridis'
    )
    plt.colorbar(sc2, ax=axes[1], label='ZeroPoint')
    axes[1].set_xlabel('Airmass')
    axes[1].set_ylabel('delta_mag (LSST - Gaia g)')
    axes[1].set_title('Airmass vs delta_mag')
    axes[1].axhline(0, color='k', lw=0.8, ls='--')

    plt.tight_layout()
    plt.show()

## 11. Requête de suivi : récupérer Visit seul (pour debug ou extension)

Si vous voulez inspecter une liste de visites précises sans refaire le JOIN complet.

In [ ]:
# Exemple : requête Visit pour une liste de visitIds
# (Reproduit le comportement du notebook RSP mais via pyvo)

def query_visits(service, visit_ids):
    """Récupère les métadonnées Visit pour une liste de visitIds."""
    visit_list = ",".join(str(v) for v in visit_ids)
    q = f"SELECT * FROM dp1.Visit WHERE visit IN ({visit_list})"
    return run_tap_query(service, q)


def query_ccdvisits(service, visit_ids):
    """Récupère les métadonnées CcdVisit pour une liste de visitIds."""
    visit_list = ",".join(str(v) for v in visit_ids)
    q = f"SELECT * FROM dp1.CcdVisit WHERE visitId IN ({visit_list})"
    return run_tap_query(service, q)


# Exemple d'utilisation :
# sample_visits = list(stardata['visit'].unique()[:10])
# df_visits  = query_visits(service, sample_visits)
# df_ccdvis  = query_ccdvisits(service, sample_visits)
# print(df_visits.columns.tolist())
print("Fonctions query_visits() et query_ccdvisits() définies.")

## 12. JOIN Source × Visit × CcdVisit sur une étoile spécifique

Version ADQL du `pd.merge` final du notebook RSP – pour une étoile identifiée par son `monsterCat_index`.

In [ ]:
def query_star_lightcurve_joined(service, ra_star, dec_star, radius_arcsec=1.0, band='g'):
    """
    Récupère la courbe de lumière d'une étoile avec métadonnées
    Visit et CcdVisit via JOIN ADQL.
    
    Parameters
    ----------
    service : pyvo.dal.TAPService
    ra_star, dec_star : float – coordonnées de l'étoile (degrés)
    radius_arcsec : float – rayon de recherche en arcsec
    band : str – filtre photométrique
    """
    radius_deg = radius_arcsec / 3600.0
    
    query = f"""
    SELECT
        src.sourceId,
        src.visit,
        src.detector,
        src.ra,
        src.dec,
        src.psfFlux,
        src.psfFluxErr,
        src.band,
        vis.expMidptMJD,
        vis.airmass,
        vis.obsStart,
        ccd.seeing,
        ccd.skyBg,
        ccd.skyNoise,
        ccd.zeroPoint,
        ccd.magLim,
        ccd.obsStartMJD AS obsStartMJD_det,
        ccd.azimuth
    FROM
        dp1.Source AS src
        JOIN dp1.Visit    AS vis ON src.visit    = vis.visit
        JOIN dp1.CcdVisit AS ccd ON src.visit    = ccd.visitId
                                 AND src.detector = ccd.detector
    WHERE
        CONTAINS(
            POINT('ICRS', src.coord_ra, src.coord_dec),
            CIRCLE('ICRS', {ra_star}, {dec_star}, {radius_deg})
        ) = 1
        AND src.band = '{band}'
    ORDER BY vis.expMidptMJD ASC
    """
    return run_tap_query(service, query)


print("Fonction query_star_lightcurve_joined() définie.")
print("Usage : df = query_star_lightcurve_joined(service, ra=53.12, dec=-28.05)")

In [ ]:
# Exemple : courbe de lumière d'une étoile du catalogue
# (décommenter et adapter les coordonnées)

# RA_STAR  = 53.05
# DEC_STAR = -28.10
# df_star_lc = query_star_lightcurve_joined(service, RA_STAR, DEC_STAR, radius_arcsec=0.5)
# print(df_star_lc.shape)
# df_star_lc.head()

print("Décommenter le bloc ci-dessus et renseigner les coordonnées pour récupérer une LC.")

## 13. Fonction utilitaire : binned running median + MAD

In [ ]:
def binned_running_median_mad(x, y, bins=30, plot=True, ax=None, **plot_kwargs):
    """
    Calcule et trace la médiane courante et le MAD de y dans des bins de x.
    
    Returns
    -------
    bin_centers, medians, mads, ax
    """
    x = np.asarray(x)
    y = np.asarray(y)
    
    medians, bin_edges, _ = binned_statistic(x, y, statistic='median', bins=bins)
    bin_centers = 0.5 * (bin_edges[1:] + bin_edges[:-1])
    
    mads = []
    for i in range(len(bin_edges) - 1):
        mask = (x >= bin_edges[i]) & (x < bin_edges[i + 1])
        mads.append(median_abs_deviation(y[mask], scale='normal') if np.any(mask) else np.nan)
    mads = np.array(mads)
    
    if plot:
        if ax is None:
            _, ax = plt.subplots()
        color = plot_kwargs.pop('color', 'C0')
        ax.plot(bin_centers, medians, color=color, label='Median', **plot_kwargs)
        ax.fill_between(bin_centers, medians - mads, medians + mads,
                        color=color, alpha=0.3, label='±MAD')

    return bin_centers, medians, mads, ax

print("Fonction binned_running_median_mad() définie.")

## 14. Résumé du schéma de tables DP1 utilisé

```
dp1.Source
  sourceId, visit, detector, coord_ra, coord_dec,
  ra, dec, psfFlux, psfFluxErr, band, ...
        │
        ├── JOIN ON visit = dp1.Visit.visit
        │      expMidptMJD, airmass, ra, dec, obsStart, band, ...
        │
        └── JOIN ON (visit, detector) = (dp1.CcdVisit.visitId, dp1.CcdVisit.detector)
               seeing, skyBg, skyNoise, zeroPoint, magLim, azimuth, obsStartMJD, ...
```

### Notes importantes
- `dp1.Source.visit` = `dp1.Visit.visit` = `dp1.CcdVisit.visitId`  
- `dp1.CcdVisit` a une ligne par **(visitId, detector)** → le JOIN doit porter sur les 2 colonnes  
- Les colonnes `seeing`, `skyBg`, `zeroPoint`, etc. dans `CcdVisit` varient par CCD, pas seulement par visite

In [ ]:
# Exploration des tables disponibles
print("Tables dp1.* disponibles :")
table_names = [n for n in service.tables.keys() if n.startswith('dp1.')]
for name in sorted(table_names):
    print(" ", name)

In [ ]:
# Colonnes de CcdVisit (pour référence)
if 'dp1.CcdVisit' in service.tables:
    ccdvisit_table = service.tables['dp1.CcdVisit']
    print("Colonnes dp1.CcdVisit :")
    for col in ccdvisit_table.columns:
        print(f"  {col.name:30s}  {col.description or ''}")

In [ ]:
df_sources.to_csv(SOURCES_FILE) 